# 工业磨机、EV Critic 核验
同一第 1900 回合固定模型；55 天回放，每类 48 个可控状态。完整口径和限制见 README.md。

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np
audit_dir = next(root / 'reports/dr_critic_audit_20261007' for root in [Path.cwd(), *Path.cwd().parents] if (root / 'reports/dr_critic_audit_20261007/audit_summary.json').exists())
summary = json.loads((audit_dir / 'audit_summary.json').read_text(encoding='utf-8'))
pd.DataFrame(summary['results'])

In [ ]:
probes = pd.read_csv(audit_dir / 'counterfactual_probes.csv')
checks = {}
for agent, group in probes.groupby('agent'):
    local = group[np.isclose(abs(group.power_delta_kw), 1, atol=1e-5, rtol=0)]
    meaningful = local[local.gain.abs() > .001]
    checks[agent] = (len(local), len(meaningful), int(sum(meaningful.dq1 * meaningful.gain < 0)))
assert checks == {'mg1': (50, 26, 6), 'mg3': (48, 15, 8)}
checks

In [ ]:
costs = [c for c in probes if c.startswith('delta_cost_')]
assert np.max(abs(probes.gain + probes[costs].sum(axis=1) / 100)) < 1e-10
summary['checks']

In [ ]:
wrong = probes[(probes.gain.abs() > .001) & (probes.dq1 * probes.gain < 0)]
wrong[['agent','date','hour','reference_kw','candidate_kw','gain','dq1','dq2','service_no_worse']].head(15)

In [ ]:
summary['caveats']